# EHR2Path original sample walkthrough

目的：用一个真实的 EHR2Path generated sample，逐步展示数据如何从 `.json` 记录变成 structured text，再进入 chat tokenizer / label mask。

本 notebook **只读文件、不写训练数据、不输出未脱敏样本文件**。所有患者/住院/ICU stay ID 均在展示前 redacted。

对应源码入口：

```text
/home/vanila/code/ehrtopath/patient_model/dataset.py
  TextDataset.__getitem__
  generate_and_tokenize_prompt
```

EHR2Path 的核心输入形式：

```text
input  = YAML(sample["desc"])
target = YAML(sample["change_log"])
```

In [1]:
from pathlib import Path
import csv
import gzip
import json
import re
import textwrap
from typing import Any

try:
    import yaml
except ImportError as e:
    raise RuntimeError("This notebook needs PyYAML: pip install pyyaml") from e

MIMIC_ROOT = Path('/mnt/d/Data/mimic-iv-2.2')
EHR2PATH_DATA_DIR = MIMIC_ROOT / 'all_data_24_hours_los_noisy'
COHORT_CSV = Path('/mnt/d/Data/Data decision output/mimiciv_trauma_cohort.csv')
DATASET_PY = Path('/home/vanila/code/ehrtopath/patient_model/dataset.py')
RETRIEVE_CHANGE_LOG_PY = Path('/home/vanila/code/ehrtopath/patient_model/retrieve_change_log.py')

print('MIMIC_ROOT exists:', MIMIC_ROOT.exists())
print('EHR2PATH_DATA_DIR exists:', EHR2PATH_DATA_DIR.exists())
print('COHORT_CSV exists:', COHORT_CSV.exists())
print('DATASET_PY exists:', DATASET_PY.exists())

MIMIC_ROOT exists: True
EHR2PATH_DATA_DIR exists: True
COHORT_CSV exists: True
DATASET_PY exists: True


### 这个 cell 做了什么

导入标准库（csv, json, gzip, yaml 等）并验证四个关键路径存在：

| 变量 | 指向 |
|---|---|
| `MIMIC_ROOT` | MIMIC-IV 原始数据根目录 `/mnt/d/Data/mimic-iv-2.2` |
| `EHR2PATH_DATA_DIR` | EHR2Path 预生成的 JSON sample 目录 |
| `COHORT_CSV` | trauma cohort CSV，用于选一个和 cohort 重叠的 sample |
| `DATASET_PY` | EHR2Path 的 `TextDataset` 源码文件 |

这些路径在后面 cell 里会被反复使用。

## 1. 读取 EHR2Path 源码片段：`TextDataset.__getitem__`

下面这个 cell 会把 `dataset.py` 的第 260-328 行打印出来。

这就是 EHR2Path 的核心 sample 读取逻辑。我们接下来要一步步复刻这些代码的效果。

关键行号速查：

| 行号 | 含义 |
|---|---|
| 260 | `class TextDataset(Dataset):` 类定义 |
| 302 | `def __getitem__(self, idx):` 入口 |
| 304 | `sample_path, timepoint_idx = sample_info.rsplit("_", 1)` 拆出文件名和时间点 |
| 306 | `sample = json.load(open(...))` 读取 JSON |
| 315 | `sample = sample[int(timepoint_idx)]` 取这个时间点的记录 |
| 318 | `sample["desc"] = summerize_patient_lvl_4(...)` 控制输入长度 |
| 319 | `sample["change_log"] = summerize_patient_lvl_4(...)` 控制 target 长度 |
| 323 | `input = yaml.dump(sample["desc"], ...)` 结构体 → YAML 文本 |
| 324 | `output = yaml.dump(sample["change_log"], ...)` 同上 |
| 326 | `prompt = generate_and_tokenize_prompt(input, output, tokenizer, ...)` 进入 tokenizer |

In [2]:
def print_source(path: Path, start: int, end: int):
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f'{i:4d}| {lines[i-1]}')

print('--- dataset.py: TextDataset.__getitem__ ---')
print_source(DATASET_PY, 260, 328)

--- dataset.py: TextDataset.__getitem__ ---
 260| class TextDataset(Dataset):
 261|     def __init__(self, tokenizer, split='train', predict=False, max_input_len=4000, max_output_len=1000,
 262|                  model_name="Meta-Llama-3.1-8B-Instruct-bnb-4bit", weighted_sampling=False, custom_attn_mask=False, dataset_name="24h_los",
 263|                  predict_all=False, add_gen_prompt_for_predict=True):
 264|         self.add_gen_prompt_for_predict = add_gen_prompt_for_predict
 265|         if split == "train" and os.environ.get("EHR2PATH_VAL_ONLY_NO_TRAIN") == "1":
 266|             print("EHR2PATH_VAL_ONLY_NO_TRAIN=1: redirecting train split to val metadata")
 267|             split = "val"
 268|         # list files in all_data
 269|         if weighted_sampling == True and split == "train":
 270|             with gzip.open(f"{mimiciv_path}{split}_weighting_sample_timepoint_paths_1Mio.json.gz", "rt") as f:
 271|                 self.data = json.load(f)
 272|         else:  # for

### 这个 cell 做了什么

直接打印了 `dataset.py` 的 `TextDataset.__getitem__` 方法源码。

你应该能看到：
- 如何从 `sample_info` 拆分出 `sample_path` 和 `timepoint_idx`
- 如何 `json.load(open(...))` 读取预生成的 JSON 文件
- 如何 `yaml.dump()` 把 structured dict 转成纯文本
- 最后调用 `generate_and_tokenize_prompt()` 交给 tokenizer

这就是 EHR2Path 的 **唯一数据入口**：一切从这 60 行代码开始。

### 再看 `generate_and_tokenize_prompt` 的 label mask 逻辑

下一个 cell 打印第 100-124 行，这段代码处理 **哪些 token 参与 loss 计算**。

In [3]:
print('--- dataset.py: generate_and_tokenize_prompt label mask ---')
print_source(DATASET_PY, 100, 124)

--- dataset.py: generate_and_tokenize_prompt label mask ---
 100|     if model_name == "Meta-Llama-3.1-8B-Instruct-bnb-4bit":
 101|         messages_full = [
 102|             {"role": "system", "content": "You are a bot for predicting the changes in a patient state in the next hour."},
 103|             {"role": "user", "content": input},
 104|             {"role": "assistant", "content": output}
 105|         ]
 106|     else:  # qwen
 107|         messages_full = [
 108|             {"role": "user", "content": input},
 109|             {"role": "assistant", "content": output}
 110|         ]
 111| 
 112|     tokenized_full_prompt = tokenizer.apply_chat_template(messages_full, add_generation_prompt=False, return_dict=True)
 113| 
 114|     if not train_on_inputs:
 115|         user_messages = messages_full[:-1]
 116|         tokenized_user_prompt = tokenizer.apply_chat_template(user_messages, add_generation_prompt=True, return_dict=False)
 117|         user_prompt_len = len(tokenized

### 这个 cell 做了什么

打印了 `generate_and_tokenize_prompt()` 中 label mask 的核心代码。

关键逻辑（第 114-122 行）：

```python
user_messages = messages_full[:-1]           # 去掉 assistant 消息
tokenized_user_prompt = tokenizer.apply_chat_template(user_messages, ...)
user_prompt_len = len(tokenized_user_prompt)

tokenized_full_prompt["labels"] = (
    [-100] * user_prompt_len                # system+user 部分：不参与 loss
    + tokenized_full_prompt["input_ids"][user_prompt_len:]  # assistant 部分：参与 loss
)
```

`-100` 是 PyTorch cross-entropy 的 ignore_index。模型不会学习复述 `desc`，只学习生成 `change_log`。

## 2. 查看 split metadata 的形状

EHR2Path 不是直接遍历 JSON 文件。它先读取 **split metadata**，这些文件记录了：

```text
train_weighting_sample_timepoint_paths_1Mio.json.gz    # 训练集，100 万条
val_noweighting_sample_timepoint_paths.json.gz          # 验证集
test_noweighting_sample_timepoint_paths.json.gz         # 测试集
```

每个 entry 形如：

```text
stay_<subject_id>_<hadm_id>.json_<timepoint_key>
```

例如 `stay_17640354_20483724.json_342` 表示：
- 文件名：`stay_17640354_20483724.json`
- 时间点 key：`342`

在源码 `TextDataset.__init__` 里（`dataset.py:270-274`）：

```python
with gzip.open(f"{mimiciv_path}{split}_...json.gz", "rt") as f:
    self.data = json.load(f)
```

下面这个 cell 打开这三个 metadata 文件，看它们的条目数和前几条。

In [4]:
for name in [
    'train_weighting_sample_timepoint_paths_1Mio.json.gz',
    'val_noweighting_sample_timepoint_paths.json.gz',
    'test_noweighting_sample_timepoint_paths.json.gz',
]:
    p = MIMIC_ROOT / name
    print('\n', name)
    print('exists:', p.exists(), 'bytes:', p.stat().st_size if p.exists() else None)
    if p.exists():
        arr = json.load(gzip.open(p, 'rt'))
        print('n_entries:', len(arr))
        print('first_3_redacted:')
        for x in arr[:3]:
            sample_path, timepoint_idx = x.rsplit('_', 1)
            redacted_path = re.sub(r'\d{5,}', '<ID>', sample_path)
            print('  sample_path=', redacted_path, 'timepoint_idx=', timepoint_idx)


 train_weighting_sample_timepoint_paths_1Mio.json.gz
exists: True bytes: 11245221
n_entries: 1000000
first_3_redacted:
  sample_path= stay_<ID>_<ID>.0.json timepoint_idx= 25
  sample_path= stay_<ID>_<ID>.0.json timepoint_idx= 20
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 10

 val_noweighting_sample_timepoint_paths.json.gz
exists: True bytes: 606329
n_entries: 62295
first_3_redacted:
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 66
  sample_path= stay_<ID>_<ID>.0.json timepoint_idx= 6
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 83

 test_noweighting_sample_timepoint_paths.json.gz
exists: True bytes: 630021
n_entries: 64659
first_3_redacted:
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 70
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 7
  sample_path= stay_<ID>_<ID>.json timepoint_idx= 41


### 这个 cell 做了什么

打开三个 `*_sample_timepoint_paths.json.gz`：

1. 打印每个文件的条目数
2. 打印前 3 条 entry，拆出 `sample_path` 和 `timepoint_idx`

你应该能看到：
- 训练集 ~100 万条
- 验证集 ~6.2 万条
- 测试集 ~6.5 万条

每个 entry 都是一条可训练的 sample（一个患者 + 一个时刻）。

## 3. 从 trauma cohort 中选一个 EHR2Path sample

这一步做两件事：

1. **定义 `redact()` 函数**：递归遍历 dict/list，把 `subject_id / hadm_id / stay_id` 以及 5 位以上纯数字替换为 `<ID_REDACTED>`，长字符串截断。后续所有展示都用它脱敏。

2. **`find_trauma_ehr2path_sample()` 函数**：遍历 `mimiciv_trauma_cohort.csv`，对每一行的 `subject_id` 和 `hadm_id`，检查 EHR2Path 生成目录里是否有对应的 `stay_<subject>_<hadm>.json` 文件。找到第一个包含 ICU Stay 内容、且 change_log 非空的 timepoint 就返回。

> 注意：这个函数内部使用了患者的原始 ID 做本地文件查找，但返回值 `sample_info` 会被 redacted 后才打印。

In [5]:
def redact_scalar(x: Any):
    if x is None or isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return x
    s = str(x)
    if re.fullmatch(r'\d{5,}', s):
        return '<ID_REDACTED>'
    if len(s) > 100:
        return s[:97] + '...'
    return s

def redact(obj: Any, depth: int = 0, max_depth: int = 5, max_items: int = 8):
    if depth >= max_depth:
        return '<TRUNCATED_DEPTH>'
    if isinstance(obj, dict):
        out = {}
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items:
                out['<...>'] = f'{len(obj)-max_items} more keys'
                break
            rk = str(k)
            if rk.lower() in {'subject_id', 'hadm_id', 'stay_id'} or re.fullmatch(r'\d{5,}', rk):
                rk = '<ID_REDACTED_KEY>'
            out[rk] = redact(v, depth + 1, max_depth=max_depth, max_items=max_items)
        return out
    if isinstance(obj, list):
        out = [redact(v, depth + 1, max_depth=max_depth, max_items=max_items) for v in obj[:max_items]]
        if len(obj) > max_items:
            out.append(f'<... {len(obj)-max_items} more items>')
        return out
    return redact_scalar(obj)

def find_trauma_ehr2path_sample():
    with COHORT_CSV.open(newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            # EHR2Path generated filenames are subject/hadm based.
            # We use IDs only for local lookup, never display them.
            candidates = [
                EHR2PATH_DATA_DIR / f"stay_{row['subject_id']}_{row['hadm_id']}.json",
                EHR2PATH_DATA_DIR / f"stay_{row['subject_id']}_{row['hadm_id']}.0.json",
            ]
            for p in candidates:
                if not p.exists():
                    continue
                obj = json.load(open(p, encoding='utf-8'))
                items = obj.items() if isinstance(obj, dict) else enumerate(obj)
                for tp_key, rec in items:
                    if 'ICU Stay' in rec.get('desc', {}) and rec.get('change_log'):
                        sample_info = f'{p.name}_{tp_key}'
                        return sample_info, p, tp_key, rec
    raise RuntimeError('No overlapping trauma/EHR2Path sample found.')

sample_info, selected_path, selected_timepoint_key, selected_record_direct = find_trauma_ehr2path_sample()
print('sample_info_redacted:', re.sub(r'\d{5,}', '<ID>', sample_info))
print('selected_file_redacted:', re.sub(r'\d{5,}', '<ID>', selected_path.name))
print('selected_timepoint_key:', selected_timepoint_key)

sample_info_redacted: stay_<ID>_<ID>.json_342
selected_file_redacted: stay_<ID>_<ID>.json
selected_timepoint_key: 342


### 这个 cell 做了什么

运行 `find_trauma_ehr2path_sample()`，打印：

| 输出 | 含义 |
|---|---|
| `sample_info_redacted` | `stay_<ID>_<ID>.json_<timepoint_key>` |
| `selected_file_redacted` | 被选中的 JSON 文件名 |
| `selected_timepoint_key` | 具体用哪个时间点 |

此时还没有加载 JSON 内容，只是定位到了哪个文件 + 哪个 timepoint。

## 4. 复刻 `TextDataset.__getitem__` 的 sample 加载

下面严格按 EHR2Path 源码的三步走：

```python
# 1. 拆分 metadata entry
sample_path, timepoint_idx = sample_info.rsplit('_', 1)

# 2. 读取 JSON 文件
sample = json.load(open(...))

# 3. 取出对应 timepoint 的记录
sample = sample[timepoint_idx]
```

注意：`rsplit('_', 1)` 是从**右侧**第一个 `_` 拆开，因为文件名本身可能也包含 `_`。

In [6]:
sample_path, timepoint_idx = sample_info.rsplit('_', 1)
loaded = json.load(open(EHR2PATH_DATA_DIR / sample_path, encoding='utf-8'))
try:
    sample = loaded[timepoint_idx]
except Exception:
    sample = loaded[int(timepoint_idx)]

print('parsed_sample_path_redacted:', re.sub(r'\d{5,}', '<ID>', sample_path))
print('parsed_timepoint_idx:', timepoint_idx)
print('record_top_keys:', list(sample.keys()))
print('hour_idx:', sample.get('hour_idx'))
print('stay_id:', '<ID_REDACTED>' if sample.get('stay_id') is not None else None)
print('desc_top_sections:', list(sample.get('desc', {}).keys()))
print('change_log_top_sections:', list(sample.get('change_log', {}).keys()))

parsed_sample_path_redacted: stay_<ID>_<ID>.json
parsed_timepoint_idx: 342
record_top_keys: ['desc', 'change_log', 'stay_id', 'hour_idx']
hour_idx: 341
stay_id: <ID_REDACTED>
desc_top_sections: ['Emergency Department Stay', 'Hospital Stay', 'ICU Stay']
change_log_top_sections: ['Hospital Stay', 'ICU Stay']


### 这个 cell 做了什么

读取了实际的 JSON 文件，取出了对应 timepoint 的记录。

你应该能看到：

| 输出 | 含义 |
|---|---|
| `record_top_keys` | `['desc', 'change_log', 'stay_id', 'hour_idx']` — 每条 record 的四个顶层字段 |
| `hour_idx` | 当前时刻是第几小时（相对于 ICU 入院） |
| `desc_top_sections` | `desc` 里有哪些 section，这里是 `Emergency Department Stay / Hospital Stay / ICU Stay` |
| `change_log_top_sections` | `change_log` 里有哪些 section |

`desc` = input，`change_log` = target。这是 EHR2Path 的 sample 本质。

## 5. 查看 `desc`：当前结构化病人状态

`desc` 是 EHR2Path 定义的 "当前时刻病人状态"。它是一个 nested dict，按 **section-first** 组织：

```text
Emergency Department Stay    ← ED 信息（入院前）
  ├── General
  ├── Chief Complaint
  ├── Admission Vitals
  └── Diagnosis

Hospital Stay                ← 住院层信息
  ├── General
  ├── Patient Location
  ├── Care Taker
  ├── Lab Results
  ├── Microbiology
  ├── Prescriptions
  └── Procedures

ICU Stay                      ← ICU 层信息
  └── Stay 0
      ├── Medication
      ├── Output
      ├── Procedures
      └── Chart Events
          ├── RoutineVitalSigns  (HR, BP, RR...)
          ├── Respiratory
          ├── Neurological
          └── ...
```

下面这个 cell 用 `redact()` 脱敏后打印 JSON 结构。

In [7]:
desc = sample['desc']
print(json.dumps(redact(desc, max_depth=4, max_items=6), ensure_ascii=False, indent=2)[:7000])

{
  "Emergency Department Stay": {
    "General": "unknown patient, male, unknown",
    "Chief Complaint": "ich, transfer",
    "Admission Vitals": {
      "pain": "Critical/10"
    },
    "Diagnosis": "traum subrac hem w/o loss of consciousness, init, fall on same level, unspecified, initial encounter"
  },
  "Hospital Stay": {
    "General": "ew emer. patient, 54-year old male, insurance: Medicare, unknown, language: english",
    "Patient Location": "24-0: Neuro Surgical Intensive Care Unit (Neuro SICU)",
    "Care Taker": "23-0: Neurologic Surgical",
    "Lab Results": {
      "Anion Gap (mEq/L, normal range: 8.0-20.0)": "11: 18, 0: 16",
      "Bicarbonate (mEq/L, normal range: 22.0-32.0)": "11: 20, 0: 19",
      "Calcium, Total (mg/dL, normal range: 8.4-10.3)": "11: 8.2, 0: 8.1",
      "Chloride (mEq/L, normal range: 96.0-108.0)": "11: 106, 0: 108",
      "Creatinine (mg/dL, normal range: 0.5-1.2)": "11/0: 0.5",
      "Glucose (mg/dL, normal range: 70.0-100.0)": "11: 106, 0: 114",

### 这个 cell 做了什么

打印了 `desc` 的 redacted JSON，你可以看到：

- EHR2Path 把 Lab Results 编码成紧凑字符串，如 `'Anion Gap (mEq/L...): 11: 18, 0: 16'`
- 数值和正常范围混在一起
- 时间信息用 `小时:值` 的格式内联在字段值里

这不是我们 EHR-Predict 想要的 canonical 变量格式，但它展示了 EHR2Path 的 structured text 设计思路。

## 6. 查看 `change_log`：下一小时 structured target

`change_log` 是训练目标 — 模型要预测 "下一小时发生了哪些变化"。

它不是单个数值或 label，而是一个 **structured dict**，内容格式与 `desc` 类似但更精简：

```yaml
Hospital Stay:
  LOS: 1085 hours
  Patient Location: Neuro SICU
  Prescriptions: [propofol, potassium chloride, ...]

ICU Stay:
  Stay 0:
    Chart Events:
      RoutineVitalSigns:
        Heart Rate: 99.0
        Arterial Blood Pressure systolic: 141.0
      Respiratory:
        O2 saturation pulseoxymetry: 100.0
```

下面打印 `change_log` 的 redacted 结构。

In [8]:
change_log = sample['change_log']
print(json.dumps(redact(change_log, max_depth=4, max_items=6), ensure_ascii=False, indent=2)[:7000])

{
  "Hospital Stay": {
    "LOS": "1085 hours",
    "Patient Location": "Neuro Surgical Intensive Care Unit (Neuro SICU)",
    "Care Taker": "NSURG",
    "Prescriptions": [
      "propofol",
      "potassium chloride",
      "magnesium sulfate in water",
      "sodium chloride",
      "acetaminophen",
      "vancomycin hydrochloride",
      "<... 15 more items>"
    ]
  },
  "ICU Stay": {
    "Stay 0": {
      "LOS": "743 hours",
      "Medication": [
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>"
      ],
      "Output": {
        "Cerebral Ventricular #1": "<TRUNCATED_DEPTH>",
        "Foley": "<TRUNCATED_DEPTH>",
        "Gastric Tube": "<TRUNCATED_DEPTH>"
      },
      "Procedures": [
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>",
        "<TRUNCATED_DEPTH>"
      ],
      "Chart Events": {
        "RoutineVitalSigns": "<TRUNCATED_DEPTH>",
      

### 这个 cell 做了什么

打印了 `change_log` 的 redacted JSON。

你应该注意到：
- `change_log` 比 `desc` 小得多 — 它只描述**下一小时**的变化
- 数值变成了单值（如 `Heart Rate: 99.0`），不再是时间序列
- 包含了 LOS、位置、用药、生命体征等

这是训练时模型要**生成**的内容。如果模型输出这些 YAML，就说明它学会了预测下一小时的状态变化。

## 7. dict → YAML 文本

EHR2Path 把 `desc` 和 `change_log` 从 nested dict 转成 YAML 字符串后，才喂给 tokenizer。

源码（`dataset.py:322-324`）：

```python
yaml.add_representer(float, self.float_representer, Dumper=yaml.SafeDumper)
input = yaml.dump(sample["desc"], sort_keys=False, default_style="'", Dumper=yaml.SafeDumper)
output = yaml.dump(sample["change_log"], sort_keys=False, default_style="'", Dumper=yaml.SafeDumper)
```

注意：
- `float_representer` 会把所有浮点数转成**字符串**（加引号），防止精度丢失
- `default_style="'"` 让所有值都带单引号
- 排序关闭 (`sort_keys=False`)，保留原始字段顺序

下面这个 cell 对 redacted 的 `desc` 和 `change_log` 做同样的 YAML dump。

In [9]:
input_yaml = yaml.dump(redact(desc, max_depth=6, max_items=8), sort_keys=False, default_style="'", Dumper=yaml.SafeDumper)
output_yaml = yaml.dump(redact(change_log, max_depth=6, max_items=8), sort_keys=False, default_style="'", Dumper=yaml.SafeDumper)

print('--- input_yaml first 120 lines ---')
print('\n'.join(input_yaml.splitlines()[:120]))
print('\n--- output_yaml first 90 lines ---')
print('\n'.join(output_yaml.splitlines()[:90]))

--- input_yaml first 120 lines ---
'Emergency Department Stay':
  'General': 'unknown patient, male, unknown'
  'Chief Complaint': 'ich, transfer'
  'Admission Vitals':
    'pain': 'Critical/10'
  'Diagnosis': 'traum subrac hem w/o loss of consciousness, init, fall on same level,
    unspecified, initial encounter'
'Hospital Stay':
  'General': 'ew emer. patient, 54-year old male, insurance: Medicare, unknown, language:
    english'
  'Patient Location': '24-0: Neuro Surgical Intensive Care Unit (Neuro SICU)'
  'Care Taker': '23-0: Neurologic Surgical'
  'Lab Results':
    'Anion Gap (mEq/L, normal range: 8.0-20.0)': '11: 18, 0: 16'
    'Bicarbonate (mEq/L, normal range: 22.0-32.0)': '11: 20, 0: 19'
    'Calcium, Total (mg/dL, normal range: 8.4-10.3)': '11: 8.2, 0: 8.1'
    'Chloride (mEq/L, normal range: 96.0-108.0)': '11: 106, 0: 108'
    'Creatinine (mg/dL, normal range: 0.5-1.2)': '11/0: 0.5'
    'Glucose (mg/dL, normal range: 70.0-100.0)': '11: 106, 0: 114'
    'H': '11: 48, 0: 30

### 这个 cell 做了什么

打印了 `desc` 和 `change_log` 各自的 YAML 文本（各截断到 ~120/90 行）。

你应该能看到：
- YAML 层级结构由缩进表示
- 所有值都用单引号包裹
- 这串 YAML 文本接下来会直接塞进 chat message 的 `content` 字段

这就是 LLM 实际看到的 "文本输入"。

## 8. 构造 chat messages

EHR2Path 使用标准的 `transformers.apply_chat_template`，构造 messages 列表：

```python
messages_full = [
    {"role": "system", "content": "You are a bot for predicting..."},
    {"role": "user",    "content": input_yaml},
    {"role": "assistant","content": output_yaml},
]
```

不同 model 族的 system prompt 处理方式不同：

| 模型 | messages 构造 |
|---|---|
| Llama 3.1 | 有 system role |
| Qwen | 无 system role，只有 user/assistant |

下面这个 cell 构造 messages 并打印每个 message 的 role 和前 300 字符。

In [10]:
SYSTEM_PROMPT = 'You are a bot for predicting the changes in a patient state in the next hour.'
messages_full = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': input_yaml},
    {'role': 'assistant', 'content': output_yaml},
]

for i, m in enumerate(messages_full):
    print(f"message[{i}] role={m['role']} chars={len(m['content'])}")
    print(m['content'][:300].replace('\n', '\\n'))
    print()

message[0] role=system chars=77
You are a bot for predicting the changes in a patient state in the next hour.

message[1] role=user chars=7694
'Emergency Department Stay':\n  'General': 'unknown patient, male, unknown'\n  'Chief Complaint': 'ich, transfer'\n  'Admission Vitals':\n    'pain': 'Critical/10'\n  'Diagnosis': 'traum subrac hem w/o loss of consciousness, init, fall on same level,\n    unspecified, initial encounter'\n'Hospital Stay':\n 

message[2] role=assistant chars=1952
'Hospital Stay':\n  'LOS': '1085 hours'\n  'Patient Location': 'Neuro Surgical Intensive Care Unit (Neuro SICU)'\n  'Care Taker': 'NSURG'\n  'Prescriptions':\n  - 'propofol'\n  - 'potassium chloride'\n  - 'magnesium sulfate in water'\n  - 'sodium chloride'\n  - 'acetaminophen'\n  - 'vancomycin hydrochloride'\n



### 这个 cell 做了什么

构造了三个 messages（system, user, assistant），打印各自角色和长度。

最终在训练/推理时，这三个 messages 会被 `tokenizer.apply_chat_template()` 展开成带特殊 token 的完整 token 序列，例如：

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a bot...
<|eot_id|><|start_header_id|>user<|end_header_id|>
YAML(desc)...
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
YAML(change_log)...
<|eot_id|>
```

## 9. Label mask 边界

训练时，EHR2Path **不训练模型复述输入**。它只对 assistant / `change_log` 部分计算 loss。

源码逻辑（`dataset.py:114-122`）：

```python
# 取 user_messages = [system, user]  #[1m](不包含 assistant)[0m
tokenized_user_prompt = tokenizer.apply_chat_template(user_messages, ...)
user_prompt_len = len(tokenized_user_prompt)

# labels: 前 user_prompt_len 个位置填 -100，后面填真实 token id
tokenized_full_prompt["labels"] = (
    [-100] * user_prompt_len
    + tokenized_full_prompt["input_ids"][user_prompt_len:]
)
```

下面这个 cell 用**字符级**模拟（不需要真实 tokenizer）来可视化这条边界。

In [11]:
def explain_label_mask_without_real_tokenizer(messages):
    # This is a conceptual visualization only. Real EHR2Path uses HF tokenizer.apply_chat_template.
    pseudo_user_prompt = '<system>\n' + messages[0]['content'] + '\n<user>\n' + messages[1]['content'] + '\n<assistant>\n'
    pseudo_full = pseudo_user_prompt + messages[2]['content']
    user_chars = len(pseudo_user_prompt)
    full_chars = len(pseudo_full)
    target_chars = full_chars - user_chars
    print('Conceptual character boundary, not real token count:')
    print('  user/system chars masked:', user_chars)
    print('  assistant target chars:', target_chars)
    print('  full chars:', full_chars)
    print('\nLoss-bearing text begins with:')
    print(messages[2]['content'][:500])

explain_label_mask_without_real_tokenizer(messages_full)

Conceptual character boundary, not real token count:
  user/system chars masked: 7801
  assistant target chars: 1952
  full chars: 9753

Loss-bearing text begins with:
'Hospital Stay':
  'LOS': '1085 hours'
  'Patient Location': 'Neuro Surgical Intensive Care Unit (Neuro SICU)'
  'Care Taker': 'NSURG'
  'Prescriptions':
  - 'propofol'
  - 'potassium chloride'
  - 'magnesium sulfate in water'
  - 'sodium chloride'
  - 'acetaminophen'
  - 'vancomycin hydrochloride'
  - 'pantoprazole sodium'
  - 'phenylephrine hydrochloride'
  - '<... 13 more items>'
'ICU Stay':
  'Stay 0':
    'LOS': '743 hours'
    'Medication':
    - 'Dextrose 5%'
    - 'NaCl 0.9%'
    - 'Pota


### 这个 cell 做了什么

用字符数代替 token 数，展示了 label mask 的分界位置。

你应该能看到：

```text
Conceptual character boundary:
  user/system chars masked: XXXX
  assistant target chars: YYYY
```

`user/system` 部分的 labels = `-100`（不计算 loss），`assistant` 部分才计算 loss。

换成真实 tokenizer 后，原理完全一样，只是单位从字符换成 token。

## 10. 可选：用真实 HF tokenizer 跑一遍

如果本地有对应 tokenizer，设置 `TOKENIZER_PATH` 后运行这个 cell。

需要：
- `transformers` 包（ehr2path 环境已有）
- 一个本地 tokenizer 目录，例如 Llama-3.1-8B-Instruct 的 tokenizer 路径

如果不需要，跳过即可。

In [12]:
TOKENIZER_PATH = None  # e.g. '/mnt/d/Models/.../Meta-Llama-3.1-8B-Instruct'

if TOKENIZER_PATH:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)
    tokenized_full_prompt = tokenizer.apply_chat_template(messages_full, add_generation_prompt=False, return_dict=True)
    tokenized_user_prompt = tokenizer.apply_chat_template(messages_full[:-1], add_generation_prompt=True, return_dict=False)
    user_prompt_len = len(tokenized_user_prompt)
    labels = [-100] * user_prompt_len + tokenized_full_prompt['input_ids'][user_prompt_len:]
    print('n_input_ids:', len(tokenized_full_prompt['input_ids']))
    print('user_prompt_len:', user_prompt_len)
    print('n_loss_tokens:', sum(1 for x in labels if x != -100))
else:
    print('Skipped real tokenizer. Set TOKENIZER_PATH to run this cell.')


Skipped real tokenizer. Set TOKENIZER_PATH to run this cell.


### 这个 cell 做了什么

如果 `TOKENIZER_PATH` 不是 None，则：

1. 加载 HF tokenizer
2. `apply_chat_template(messages_full)` 得到完整 token 序列
3. `apply_chat_template(messages_full[:-1])` 得到 user+system 长度
4. 构造 labels（前面填 -100）

打印：
- `n_input_ids`：完整 token 序列长度
- `user_prompt_len`：输入部分长度
- `n_loss_tokens`：实际计算 loss 的 token 数

当前 `TOKENIZER_PATH=***`，所以只会打印 "Skipped..."。

## 11. 哪些能迁移到 EHR-Predict

EHR2Path 的数据处理可以拆成 **接口层** 和 **schema 层**：

| 层 | EHR2Path | 我们迁移 |
|---|---|---|
| sample 定位 | `split metadata → sample_info.rsplit → JSON file → timepoint` | 换成 `LandmarkDataset(hadm_id, anchor_hour)` |
| state 载体 | `desc / change_log` nested dict | 换成 `input_blocks / targets` |
| 序列化 | `yaml.dump()` | 保留 `StructuredTextRenderer` |
| chat 包装 | `apply_chat_template` | 同样 |
| label mask | `[-100]*user_len + target_ids` | 同样 |

**不迁移的是 schema 层**：

- EHR2Path 用 `Emergency Department Stay / Hospital Stay / ICU Stay` 按 section 切
- 我们用 `STATIC / DAY / HOUR / CXR_EVENT / TASK` 按时间切
- EHR2Path 混入了 diagnosis、microbiology、notes 等复杂自由文本
- 我们用 canonical 变量 registry + 白名单字段

## 12. Minimal Colab note

当前 walkthrough 不需要 GPU。Colab 只在后续需要真实 tokenizer/model 或训练时有意义。

如果放到 Colab，建议：

```python
from google.colab import drive
# drive.mount('/content/drive')
```

路径要改成 Drive 路径，例如：

```python
MIMIC_ROOT = Path('/content/drive/MyDrive/Data/mimic-iv-2.2')
COHORT_CSV = Path('/content/drive/MyDrive/Data/Data decision output/mimiciv_trauma_cohort.csv')
```

注意：不要把 raw MIMIC/EHR sample push 到 GitHub。Colab notebook 应只 pull code，数据留在 Drive，本 notebook 展示时仍使用 redaction。